# 01 - Data Inspection

Objective

Inspect all available datasets before feature extraction.

Datasets

- Sentinel
- Landsat
- VIIRS
- LULC
- Ganga Buffer

Outputs

- CRS
- Resolution
- Band Information
- Bounds
- Metadata


## Environment
Install the dependencies from the repository root with `pip install -r requirements.txt` before running this notebook.

In [ ]:
import os
import rasterio
import geopandas as gpd
import pandas as pd

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

BUFFER_PATH = DATA_ROOT / "boundaries" / "ganga_buffer_150km.shp"
SENTINEL_PATH = Path(os.environ.get("GANGA_SENTINEL_DIR", DATA_ROOT / "satellite" / "Sentinel2_Weekly")).expanduser()
VIIRS_PATH = Path(os.environ.get("GANGA_VIIRS_DIR", DATA_ROOT / "satellite" / "VIIRS_2015_2025_COLOR")).expanduser()
LULC_PATH = Path(os.environ.get("GANGA_LULC_DIR", DATA_ROOT / "satellite" / "LULC_MONTHLY_2015_2025")).expanduser()

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")


In [ ]:
# ==========================================
# READ GANGA BUFFER SHAPEFILE
# ==========================================

ganga_buffer = gpd.read_file(BUFFER_PATH)

print("=" * 50)
print("GANGA BUFFER INFORMATION")
print("=" * 50)

print(f"Number of Features : {len(ganga_buffer)}")
print(f"CRS                : {ganga_buffer.crs}")
print(f"Bounds             : {ganga_buffer.total_bounds}")

print("\nFirst 5 Rows:")
display(ganga_buffer.head())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8,8))

ganga_buffer.plot(ax=ax, color="lightblue", edgecolor="black")

ax.set_title("Ganga Buffer Area")

plt.show()

In [ ]:
satellite_files = sorted(SENTINEL_PATH.rglob("*.tif"))
print(f"Total TIFF files found: {len(satellite_files)}")
if not satellite_files:
    raise FileNotFoundError(
        f"No TIFF files found under {SENTINEL_PATH}. "
        "Set GANGA_SENTINEL_DIR or place the imagery in data/satellite/Sentinel2_Weekly."
    )
first_image = satellite_files[0]
print(f"First TIFF: {first_image}")

In [ ]:
first_image = satellite_files[0]

with rasterio.open(first_image) as src:

    print("=" * 60)
    print("SATELLITE IMAGE INFORMATION")
    print("=" * 60)

    print(f"Filename     : {first_image.name}")
    print(f"CRS          : {src.crs}")
    print(f"Width        : {src.width}")
    print(f"Height       : {src.height}")
    print(f"Bands        : {src.count}")
    print(f"Resolution   : {src.res}")
    print(f"Bounds       : {src.bounds}")
    print(f"Data Type    : {src.dtypes}")
    print(f"NoData Value : {src.nodata}")
    print(f"Driver       : {src.driver}")

In [ ]:
# ==========================================
# FUNCTION TO INSPECT ANY RASTER
# ==========================================

def inspect_raster(raster_path):

    with rasterio.open(raster_path) as src:

        info = {
            "Filename": raster_path.name,
            "CRS": src.crs,
            "Width": src.width,
            "Height": src.height,
            "Bands": src.count,
            "Resolution": src.res,
            "Bounds": src.bounds,
            "Data Type": src.dtypes,
            "NoData": src.nodata,
            "Driver": src.driver
        }

    return pd.DataFrame(info.items(), columns=["Property", "Value"])

In [ ]:
inspect_raster(first_image)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with rasterio.open(first_image) as src:

    image = src.read()

plt.figure(figsize=(8,8))

plt.imshow(np.moveaxis(image, 0, -1))

plt.title(first_image.name)

plt.axis("off")

plt.show()

In [ ]:
print("First 20 files:\n")

for file in satellite_files[:20]:
    print(file.name)

In [ ]:
with rasterio.open(first_image) as src:

    print("Descriptions :", src.descriptions)
    print("ColorInterp  :", src.colorinterp)

In [ ]:
with rasterio.open(first_image) as src:

    print(src.profile)

In [ ]:
with rasterio.open(first_image) as src:

    for i in range(1, src.count + 1):

        band = src.read(i)

        print(f"\n========== Band {i} ==========")

        print(f"Minimum : {np.nanmin(band)}")
        print(f"Maximum : {np.nanmax(band)}")
        print(f"Mean    : {np.nanmean(band)}")
        print(f"Std Dev : {np.nanstd(band)}")